In [ ]:
/*
========================================================
DISTRIBUTED COUNTING - SIMPLE HINGLISH NOTES
(Counting at Scale ka Pattern)
========================================================

INTRO
--------------------------------------------------------
Counting simple lagta hai jab ek database row hi write 
rate handle kar sakta ho: kuch hone pe number increment 
karo, latest value dikhao.

SCALE PE, HARD PART ADDITION NAHI HAI. Hard part hai: 
JAB EK LOGICAL COUNTER "HOT KEY" BAN JAAYE, tab bhi write 
path fast rakhna - saath mein retries, duplicate events, 
stale reads, aur failure ke baad recovery bhi handle karna.

REAL EXAMPLES:
- YouTube video views count karta
- Twitter likes/retweets count karta
- Reddit upvotes count karta
- Instagram followers count karta
- Commerce systems inventory count karte (par isko usually 
  reservation/ledger model chahiye, simple counter nahi)

MAIN SAWAL: Jab bahut saare events EK SAATH ek hi logical 
item pe hit karein, to reliably kaise count karein?

CHALLENGE: Sahi trade-off point choose karna:
- EXACT vs APPROXIMATE
- IMMEDIATE vs DELAYED
- CHEAP READS vs CHEAP WRITES

TEXT DIAGRAM:

  Event 1, Event 2, Event 3, ... Event N
       |         |        |            |
       -----------------------------------
                    |
              [Counter] <- BOTTLENECK!

ACHA INTERVIEW ANSWER: Pehle accuracy, freshness, write 
rate, aur retry behavior clarify karo. Ye decide karega 
single counter chahiye, sharded counter, async aggregation 
pipeline, ya approximate data structure.


INTERVIEW ANSWER SHAPE
--------------------------------------------------------
Interview mein seedha sketch/queue se start mat karo. 
Pehle counter ka "correctness contract" define karo:

1. Clarify karo count EXACT hona chahiye, APPROXIMATE, 
   IMMEDIATELY VISIBLE, ya thodi delay chalegi
2. HOT KEY identify karo - ek logical counter ek row/shard/
   partition ko overload kar sakta
3. Moderate write rates ke liye - atomic database increments 
   ya conditional updates use karo
4. Hot exact counters ke liye - counter shard karo, 
   aggregated totals cache karo
5. Very high throughput ke liye (stale reads allowed) - 
   events durable log mein likho, async aggregate karo. 
   Hamesha idempotency, replay, reconciliation mention karo.


COUNTING SCALE PE HARD KYU HAI?
--------------------------------------------------------
Single server + single database pe, counting easy hai. 
User post like karta, UPDATE statement chalti jo counter 
increment karti. Database locking + atomicity handle karta, 
kaam khatam.

CODE EXAMPLE:
  UPDATE posts SET like_count = like_count + 1 WHERE post_id = 123;

Ye SIMPLE ATOMIC INCREMENT hai. Database khud row lock 
leta, increment karta, phir release karta. Ye SAHI TARIKA 
hai (application code mein read-modify-write karne se 
BEHTAR).

PROBLEMS TAB SHURU HOTE JAB SCALE BADHTA HAI.

--------------------------------------------------------
1. CONCURRENCY PROBLEM
--------------------------------------------------------
Ek popular post ko 1000 likes/second mile. Har like ek 
UPDATE trigger karti SAME row pe. Kya hota hai?

TEXT DIAGRAM - LOST UPDATES:

  User 1: Read count (100) -> Write count (101)
  User 2: Read count (100) -> Write count (101)
  User 3: Read count (100) -> Write count (101)
       |
  EXPECTED: 103
  ACTUAL: 101
  LOST UPDATES!

Bina atomic updates/locking ke, "LOST UPDATES" hote hain 
(3 likes aayi, but sirf 1 count hui). Fix: database ko 
increment ATOMIC UPDATE ya ROW LOCK se serialize karne do.

BUT SERIALIZATION KI EK LIMIT HAI: har increment same row 
pe contend karta, isliye throughput utna hi fast hai jitna 
wo row lock/update/commit ho sakti.

NOTE: In-database atomic increment FAST hai kyunki lock 
sirf update ke time hold hota. Application code mein 
read-modify-write (jaisa upar dikhaya) row ko network 
round-trip ke dauran hold karta, throughput aur bhi kam ho 
jata.

Dono cases mein, jab writes row absorb karne se FAST aaye, 
requests lock ke peeche queue hoti, latency badhti, 
requests timeout hone lagte.

REAL NUMBERS: Ek "hot row" typically SAU SE LOW-THOUSANDS 
writes/second tak hi handle kar paata (database aur commit 
settings pe depend karta). Ye normal content ke liye theek 
hai, but viral item ke liye BAHUT KAM hai.

--------------------------------------------------------
2. HOT KEY PROBLEM
--------------------------------------------------------
Distributed databases data ko multiple nodes mein partition 
karte horizontal scale ke liye. But EK VIRAL POST KA COUNTER 
matlab sab writes SAME partition, SAME node, SAME row pe 
jaayenge.

TEXT DIAGRAM:

  [Distributed Database]
    Partition 1 (Normal Posts) -> 10 writes/sec
    Partition 2 (Normal Posts) -> 15 writes/sec
    Partition 3 (VIRAL POST)   -> 10,000 writes/sec !!!
    Partition 4 (Normal Posts) -> 12 writes/sec

EK partition overloaded hai jabki baaki idle baithe hain. 
Ye "HOT KEY" ya "HOT PARTITION" problem hai, aur ye 
horizontal scaling ka purpose hi khatam kar deta hai.

--------------------------------------------------------
3. CONSISTENCY vs AVAILABILITY TRADE-OFF
--------------------------------------------------------
Hot key problem kam karne ke liye, multiple replicas/
partitions pe writes accept kar sakte. But ab CONSISTENCY 
problem aa jaata. Agar 2 replicas independently increments 
accept karein, reads ko merge strategy chahiye, aur retries 
ko same event double-count nahi karna chahiye.

  Strong Consistency  -> Har read latest write dekhta, but 
                          writes slow ho sakte, network 
                          partitions mein fail ho sakte
  Eventual Consistency -> Writes fast/available hote, but 
                          reads stale data dekh sakte

COUNTING KE LIYE INTERVIEW SAWAL: Count kitna accurate hona 
chahiye? Reads kitne fresh hone chahiye? Same event 2 baar 
deliver ho to kya hoga?


COUNTING APPROACHES KA SPECTRUM
--------------------------------------------------------
Koi ek "sahi" tarika nahi hai scale pe counting ka. Sahi 
approach requirements pe depend karta.

TEXT DIAGRAM:

  EXACT REAL-TIME      -> Single Counter (Low scale only)
  EXACT DELAYED        -> Aggregated Counter (High scale, 
                            seconds delay)
  APPROXIMATE FAST     -> Probabilistic (Very high scale, 
                            tunable error)

  Requirement                    Use Case            Approach
  -------------------------------------------------------------
  Exact + Transactional         Payments,            Ledger/
                                  inventory reservation reservation 
                                                       records, NOT 
                                                       just counter
  Exact + Immediate,             Likes, quotas,       Atomic single
  moderate write rate            rate-limit state      counter
  Exact but slightly stale      Social likes, views,  Sharded 
                                  shares                counters, 
                                                        event 
                                                        aggregation
  Approximate                   Analytics, unique     HyperLogLog, 
                                  visitors, trending    Count-Min 
                                                         Sketch, 
                                                         sampling


APPROACH 1: SINGLE COUNTER WITH ATOMIC UPDATES
--------------------------------------------------------
Simplest approach jo actually moderate scale pe kaam karta.

KAISE KAAM KARTA:
Simple +1 ya -1 ke liye, database mein ATOMIC UPDATE use 
karo. Application mein value read karke, +1 karke, wapas 
write mat karo. Storage engine ko increment serialize 
karne do.

```sql
UPDATE posts
SET like_count = like_count + 1
WHERE post_id = 123;
```

Ye ek line ka atomic increment hai. Database ne khud 
lock-increment-unlock kiya, application ko kuch alag 
karne ki zaroorat nahi.

KEY-VALUE STORE MEIN: Redis ka INCR aur INCRBY atomic hote, 
memory mein chalte, isliye ek counter bahut high rate 
handle kar sakta - isiliye like/view counters ke liye ye 
common first choice hai.

MANAGED DATABASES: Cassandra native counter columns deta, 
DynamoDB item pe atomic counter updates support karta - 
isliye distributed atomic counter khud banaye bina mil 
jaata.

OPTIMISTIC CONCURRENCY - Jab naya value SIRF simple 
increment se zyada state pe depend kare, to current value+
version read karo, naya value compute karo, phir SIRF tab 
write karo jab version na badla ho.

TEXT DIAGRAM:

  Read Current Count -> Count Changed? 
     NO  -> Write New Count -> Success
     YES -> Retry

```sql
-- Current value read karo
SELECT like_count, version FROM posts WHERE post_id = 123;
-- Returns: like_count = 100, version = 42

-- Version check ke saath update
UPDATE posts
SET like_count = 101, version = 43
WHERE post_id = 123 AND version = 42;

-- Agar 0 rows affect hui, matlab kisi aur ne pehle update 
-- kar diya - RETRY karo
```

KAB KAAM KARTA:
Ye moderate contention achhe se handle karta. Atomic 
increments lost updates avoid karte, database internally 
writes serialize karta. Ye usually SAHI STARTING POINT hai 
un counters ke liye jo hot key banne wale nahi.

KAB TOOT JATA:
Jab HAZAARON writes/second SAME row pe aayein, phir se hot 
key problem aata. Database numbers increment karne se zyada 
time LOCKS COORDINATE karne mein bitata.

BEST FOR:
- Content jo shayad hi viral ho (zyada tar user content)
- Individual counter pe moderate write rates
- Exact, immediately visible counts jaha ek row/partition 
  load handle kar sake


APPROACH 2: SHARDED COUNTERS
--------------------------------------------------------
Agar ek counter bottleneck hai, to BAHUT SAARE counters use 
karo.

KAISE KAAM KARTA:
Single counter per entity ke bajaye, N COUNTER SHARDS 
banao. Har write kisi ek shard pe jaata based on distribution 
key. Total count = sab shards ka sum.

TEXT DIAGRAM:

  WRITE PATH:
    Write Request -> Choose Shard -> [Shard 0: 2341]
                                   -> [Shard 1: 2156]
                                   -> [Shard 2: 2289]
                                   -> [Shard 3: 2214]

  READ PATH:
    Read Request -> Aggregate all shards -> Total: 9000

WRITE PATH STEPS:
1. Entity X ke liye increment request receive karo
2. Well-distributed key use karke shard choose karo
3. SIRF us shard ko increment karo

DISTRIBUTION KEY IMPORTANT HAI: 
- Raw event counts (jaise views) ke liye - event_id, 
  request_id, ya random shard selection use karo, taaki ek 
  active user ek shard ko overload na kare
- One-action-per-user counts (jaise likes) ke liye - user_id 
  hash kar sakte, but sirf tab jab system idempotency bhi 
  enforce kare (same user 2 baar count na ho)

READ PATH STEPS:
1. Entity X ke liye SAB N shards query karo
2. Values sum karo
3. Result cache karo

DATA MODEL:

```sql
-- Schema
CREATE TABLE counter_shards (
    entity_id VARCHAR(64),
    shard_id INT,
    count BIGINT DEFAULT 0,
    PRIMARY KEY (entity_id, shard_id)
);

-- Write: well-distributed key se choose kiya gaya shard 
-- increment karo
UPDATE counter_shards
SET count = count + 1
WHERE entity_id = 'post_123' AND shard_id = hash(event_id) % 10;

-- Read: sab shards aggregate karo
SELECT SUM(count)
FROM counter_shards
WHERE entity_id = 'post_123';
```

Isme har entity-shard pair ka apna row hai, isliye kisi ek 
shard pe atomic UPDATE karne se row-level contention nahi 
hota, aur reads SUM query se aggregate ho jaate.

SHARDS KI SANKHYA KAISE CHOOSE KAREIN:
Zyada shards = better write distribution but expensive 
reads. Peak write rate pe depend karta.

  Peak Writes/Second   Suggested Shards
  -----------------------------------------
  < 500                1 (sharding zaroori nahi)
  500 - 2,000          10
  2,000 - 10,000       50-100
  > 10,000             100+ ya adaptive sharding

ADAPTIVE SHARDING:
Bahut saare systems SINGLE COUNTER se start karte, aur 
sirf tab shards add karte jab contention detect ho. Ye 
"cold" entities ke liye read overhead avoid karta, jabki 
"hot" entities ko zyada write capacity deta.

TEXT DIAGRAM:

  Incoming Write -> Hot Entity?
     NO  -> Single Counter
     YES -> Sharded Counter
  
  Monitor Write Rate -> Threshold cross?
     YES -> Activate Sharding
     NO  -> Keep Single

TRADE-OFFS:

  ACHA (PROS)                         BURA (CONS)
  ------------------------------------------------------
  Write load effectively distribute   Reads ko aggregation 
                                        chahiye
  Very high write rates tak scale     Implementation zyada 
                                        complex
  Exact (agar har event ek baar apply Storage overhead 
  ho aur sab shards include hon)       (N rows per hot 
                                        entity)

NOTE: Sharded counters correctness work KHATAM nahi karte. 
Retries phir bhi increments duplicate kar sakte, cached 
totals phir bhi stale ho sakte. Interview mein idempotency 
keys, event processing ke liye deduplication, aur source 
of truth se periodic reconciliation mention karo.

BEST FOR:
- Viral content (social media posts, trending items)
- Known hot keys (celebrity accounts, popular products)
- Jab exact counts chahiye but aggregation cost/thodi stale 
  cached totals tolerate kar sakein


APPROACH 3: WRITE-BEHIND (ASYNC AGGREGATION)
--------------------------------------------------------
Bahut saare count reads ko real-time accuracy nahi chahiye. 
Kuch second purana count usually likes/views/followers ke 
liye theek hai.

KAISE KAAM KARTA:
Message queue use karke write path ko count aggregation se 
DECOUPLE karo.

TEXT DIAGRAM:

  WRITE PATH (Fast):
    Write Request -> Like Service -> Likes DB
                          |
                    Message Queue

  AGGREGATION (Async):
    Message Queue -> Count Aggregator -> Count Cache

  READ PATH (Fast):
    Read Request -> Count Cache

WRITE PATH STEPS:
1. Event (like, view, etc.) primary database mein record 
   karo
2. Event ko durable log/queue mein publish karo (Kafka, 
   SQS, etc.)
3. TURANT success return karo

WARNING - DUAL-WRITE PROBLEM: Database mein likhna aur 
queue mein publish karna 2 ALAG operations hain, isliye 
beech mein crash ho to event RECORD hua but COUNT NAHI 
hua, ya COUNT hua but RECORD NAHI hua.

ISKO AVOID KARNE KE 2 CLEAN TAREEKE:
1. Database ko source-of-truth rakho, event ko usse 
   TRANSACTIONAL OUTBOX ya CDC (Change Data Capture) se 
   derive karo - taaki event tabhi publish ho jab row 
   commit ho
2. Durable event log (jaise Kafka) ko source-of-truth 
   banao, database ko usi log ka ek consumer treat karo

Dono tareeke 2 independent writes ko disagree hone se 
rokte hain.

AGGREGATION STEPS:
1. Consumers queue se events read karte
2. Memory mein counts time window pe aggregate karte 
   (e.g., 1-5 seconds)
3. Count store ko batch update karte

READ PATH STEPS:
1. Count cache (Redis) se read karo
2. Cache miss ho to, count store query karo, cache populate 
   karo

BATCHING:
10,000 individual increments/second ke bajaye, aggregator 
unhe 10 updates/second mein batch kar sakta, har ek mein 
1,000 se increment karta. Ye database load 1000x KAM kar 
deta.

  Events in 1 second:
  +1, +1, +1, +1, +1, ... (10,000 events)
       |
  Batched update:
  UPDATE counts SET count = count + 10000 WHERE entity_id = 'post_123'

ASYNC AGGREGATION EXAMPLE:

Producer ek EVENT_ID ke saath event bhejta (deduplication 
ke liye):

```python
# Producer (ya outbox relay)
kafka_producer.send("like-events", {
    "event_id": "evt_987",
    "entity_id": "post_123",
    "event_type": "like",
    "delta": 1,
    "timestamp": now()
})
```

Consumer (Count Aggregator) events ko entity ke hisab se 
buffer karta, phir har kuch second mein Redis mein atomically 
flush karta, aur Kafka offsets sirf successful flush ke 
baad commit karta:

```python
# Consumer (Count Aggregator)
buffer = defaultdict(int)
event_ids = set()

for event in kafka_consumer:
    if event["event_id"] in event_ids or dedupe_store.seen(event["event_id"]):
        continue  # Duplicate event, skip karo

    buffer[event["entity_id"]] += event["delta"]
    event_ids.add(event["event_id"])

    if time_to_flush():  # Har 5 seconds mein
        with redis.pipeline(transaction=True) as pipe:
            for entity_id, count in buffer.items():
                pipe.incrby(f"count:{entity_id}", count)
            for event_id in event_ids:
                pipe.set(f"seen:{event_id}", 1, ex=86400)
            pipe.execute()

        kafka_consumer.commit_offsets()  # SIRF flush success 
                                          # hone ke baad commit
        buffer.clear()
        event_ids.clear()
```

HANDLING CONSISTENCY (Trade-off = eventual consistency):
User post like karne ke baad, count kuch second uska action 
reflect nahi karega. Usually acceptable, but USER KE APNE 
VIEW ko special handle karna zaroori.

READ-YOUR-OWN-WRITES PATTERN:
1. User like karne ke baad, apni session/local state mein 
   store karo
2. Us user ko counts dikhate waqt, unke pending action ko 
   cached count ke upar overlay karo
3. Jab async count catch up kar jaaye, overlay hata do

Ye user ke view ko improve karta bina ye pretend kiye ki 
global count sabke liye turant consistent hai.

RECONCILIATION:
Aggregated aur cached counts time ke saath DRIFT hote. Ek 
dropped batch, consumer bug, ya deduplication se bacha hua 
duplicate, displayed count ko thoda off kar sakta.

Reconciliation isko theek karta - source-of-truth se true 
count recompute karke (usually COUNT(*) ya SUM events/likes 
table pe), cached total overwrite karta. Isko schedule pe 
chalao - hot entities ke liye zyada baar, long-tail ke liye 
kam baar, taaki chhote errors accumulate na hon indefinitely.

TRADE-OFFS:

  ACHA (PROS)                         BURA (CONS)
  ------------------------------------------------------
  Extremely high write throughput     Counts eventually 
                                        consistent hote
  Batching se database load kam       Architecture zyada 
                                        complex
  Traffic spikes smooth karta         Idempotency, 
                                        backpressure, replay, 
                                        reconciliation chahiye

NOTE: "At-least-once delivery" usually safer default hai, 
but same event 2 baar deliver ho sakta. Aggregator ko 
IDEMPOTENT hona chahiye ya events apply karne se pehle 
DEDUPLICATE karna chahiye.

BEST FOR:
- Social media metrics (likes, views, shares)
- Analytics counters
- Koi bhi count jaha kuch second delay chalega


APPROACH 4: COUNT-MIN SKETCH (APPROXIMATE COUNTING)
--------------------------------------------------------
Jab billions events count karne hon aur exact accuracy 
zaroori na ho, probabilistic data structures BAHUT KAM 
MEMORY use karte.

KAISE KAAM KARTA:
Count-Min Sketch ek probabilistic data structure hai jo 
MULTIPLE HASH FUNCTIONS aur 2D array of counters use karta.

TEXT DIAGRAM:

  Item X -> Hash 1 -> Row 1: [0,5,3,0,2,1,0,4] (position pe 
                                                  increment)
         -> Hash 2 -> Row 2: [2,0,1,6,0,3,0,2]
         -> Hash 3 -> Row 3: [0,4,0,2,5,0,1,0]

  Query X -> Same 3 hashes -> Take MINIMUM of 3 values 
          -> Estimated Count

INSERT:
1. Item ko har hash function se hash karo
2. Har resulting position pe counter increment karo

QUERY:
1. Item ko har hash function se hash karo
2. Har position pe counter dekho
3. MINIMUM value return karo (hash collisions se 
   overcounting minimize hoti hai)

PROPERTIES:
- SPACE: O(w × d) jaha w = width, d = hash functions ki 
  sankhya
- ACCURACY: Overestimate ho sakta, but BOUNDED
- ERROR RATE: Dimensions adjust karke tune kar sakte

CHHOTA sketch bhi bade stream ko configurable error bound 
ke saath track kar sakta. Har item store nahi karta, isliye 
memory FIXED rehti chahe distinct items ki sankhya badhe.

KAB USE KAREIN:
- Bahut saare distinct items (URLs, user IDs, IP addresses) 
  kitni baar aaye estimate karna
- Approximate counts acceptable hain
- Memory constrained hai
- Fast, constant-time operations chahiye

TRADE-OFFS:

  ACHA (PROS)                    BURA (CONS)
  --------------------------------------------------
  Constant memory (item count    Approximate (overestimate 
  se independent)                 ho sakta)
  O(1) insert aur query          Decrements karna hard, 
                                   usual guarantees toot 
                                   sakte
  Merge-friendly (multiple       Heavy hashing collisions 
  nodes ke sketches combine)      se accuracy kam ho sakti

BEST FOR:
- Known keys ke frequency estimates
- Heavy hitter detection (candidate tracking/top-K ke saath)
- Network traffic analysis
- Bade stream mein values kitni baar repeat hote, without 
  storing every value


APPROACH 5: HYPERLOGLOG (CARDINALITY ESTIMATION)
--------------------------------------------------------
Jab TOTAL EVENTS nahi, UNIQUE ITEMS count karne hon, 
HyperLogLog fixed-memory estimate deta chhoti error rate 
ke saath.

PROBLEM YE SOLVE KARTA:
Website ke unique visitors, video ke unique viewers, ya 
kisi service pe access karne wale unique devices count 
karna. Naive approach (har ID ek set mein store karna) 
O(n) memory maangta. Millions uniques ke saath, ye mehenga 
ho jata.

KAISE KAAM KARTA:
HyperLogLog ek probabilistic property use karta: hashed 
values mein leading zeros ki max sankhya, unique items ki 
sankhya se correlate karti.

TEXT DIAGRAM:

  Item A -> Hash -> Binary: 00101...
  Item B -> Hash -> Binary: 00001...
  Item C -> Hash -> Binary: 01001...
       |
  Track Leading Zeros
       |
  Max zeros = 4 -> Estimate ≈ 2^4 = 16 uniques

Actual algorithm multiple "REGISTERS" (shards jaisa) aur 
HARMONIC MEAN use karta variance reduce karne ke liye.

REDIS HYPERLOGLOG EXAMPLE: